# Session 1 — Toy VAMP-seq: from bin counts to abundance scores

This is the first team assignment. The data are **invented for teaching** and are not real PTEN measurements.

**Aims** (from the end of the Session 1 slides):
1. Explain what each column means.
2. Compute weighted scores for each variant.
3. Compute normalized scores for each variant: (1) subtract the median nonsense score, then (2) divide by the wild-type score (with the median nonsense score subtracted).
4. Think critically about the results.

**For each analysis, be ready to explain:**
- what you did, conceptually (in words or figures)
- how your code achieved it
- how you checked your assumptions and verified that the code does what you intended

## 0. Setup

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

Locate the data. This notebook assumes Jupyter is running with its working directory set to this notebook's folder (`code/my_code`), so the project root is two levels up. If the last line prints `False`, check `Path.cwd()`.

In [3]:
PROJDIR = Path.cwd().parent.parent
toy_path = PROJDIR / "data" / "raw_data" / "examples" / "toy_vampseq_bin_counts.csv"
toy_path.exists()

True

In [15]:
toy = pd.read_csv(toy_path)
toy

,variant,variant_type,replicate,bin1_count,bin2_count,bin3_count,bin4_count
0,WT,wild_type,1,9,42,267,376
1,p.Gly12Gly,synonymous,1,12,32,247,316
2,p.Trp18Ter,nonsense,1,221,110,41,6
3,p.Gln36Ter,nonsense,1,217,115,38,11
4,p.Tyr52Ter,nonsense,1,207,89,30,7
5,p.Leu25Pro,missense,1,190,154,68,11
6,p.Val30Asp,missense,1,124,208,146,36
7,p.Ile40Asn,missense,1,40,172,258,43
8,p.Gly46Arg,missense,1,159,17,22,181
9,p.Ser45Thr,missense,1,17,54,214,287


## Aim 1. Explain what each column means

Fill in the table in your own words. Consider: what does one **row** represent? What order are the bins in, and what does that order mean physically?

| column | meaning |
|---|---|
| `variant` | the amino acid change |
| `variant_type` | the kind of variant (the effect of the mutation)  |
| `replicate` | biological replicate. Same mutations, different samples |
| `bin1_count` | raw fluorescence values in this bin (FACS sorted by brightness) |
| `bin2_count` | " " |
| `bin3_count` | " " |
| `bin4_count` | " " |

In [5]:
# Optional: inspect the table (shape, columns, missing values, how many of each variant_type)

print(type(toy))
print(len(toy))
print(toy.shape)

<class 'pandas.DataFrame'>
24
(24, 7)


**Presentation notes**
- What I did:
- How the code works:
- How I checked:

## Aim 2. Compute weighted scores for each variant

Reference (from the slides). For variant *v* in bin *b*, using **one bin and one replicate at a time**:

$$F(v, b) = \frac{\text{count}(v, b)}{\sum_u \text{count}(u, b)}$$

Bin weights are 0.25, 0.50, 0.75, 1.00 (bin 1 to bin 4), and the weighted score is

$$W = \frac{0.25F_1 + 0.50F_2 + 0.75F_3 + 1.00F_4}{F_1 + F_2 + F_3 + F_4}$$

Questions to answer in words: Why divide by the total of the *bin* and not by the variant's own row total? Do the four frequencies for a variant have to sum to one?

In [16]:
toy

,variant,variant_type,replicate,bin1_count,bin2_count,bin3_count,bin4_count
0,WT,wild_type,1,9,42,267,376
1,p.Gly12Gly,synonymous,1,12,32,247,316
2,p.Trp18Ter,nonsense,1,221,110,41,6
3,p.Gln36Ter,nonsense,1,217,115,38,11
4,p.Tyr52Ter,nonsense,1,207,89,30,7
5,p.Leu25Pro,missense,1,190,154,68,11
6,p.Val30Asp,missense,1,124,208,146,36
7,p.Ile40Asn,missense,1,40,172,258,43
8,p.Gly46Arg,missense,1,159,17,22,181
9,p.Ser45Thr,missense,1,17,54,214,287


### Step 1: frequencies

In [17]:
bin_cols = ["bin1_count", "bin2_count", "bin3_count", "bin4_count"]
bin_totals = toy.groupby("replicate")[bin_cols].transform("sum")
freq = toy[bin_cols].div(bin_totals) #count / sum
#make vectors and attach to df
freq_cols = [c.replace("_count", "_freq") for c in bin_cols] 
toy[freq_cols] = freq
toy

,variant,variant_type,replicate,bin1_count,bin2_count,bin3_count,bin4_count,bin1_freq,bin2_freq,bin3_freq,bin4_freq
0,WT,wild_type,1,9,42,267,376,0.007500,0.042000,0.190714,0.235000
1,p.Gly12Gly,synonymous,1,12,32,247,316,0.010000,0.032000,0.176429,0.197500
2,p.Trp18Ter,nonsense,1,221,110,41,6,0.184167,0.110000,0.029286,0.003750
3,p.Gln36Ter,nonsense,1,217,115,38,11,0.180833,0.115000,0.027143,0.006875
4,p.Tyr52Ter,nonsense,1,207,89,30,7,0.172500,0.089000,0.021429,0.004375
5,p.Leu25Pro,missense,1,190,154,68,11,0.158333,0.154000,0.048571,0.006875
6,p.Val30Asp,missense,1,124,208,146,36,0.103333,0.208000,0.104286,0.022500
7,p.Ile40Asn,missense,1,40,172,258,43,0.033333,0.172000,0.184286,0.026875
8,p.Gly46Arg,missense,1,159,17,22,181,0.132500,0.017000,0.015714,0.113125
9,p.Ser45Thr,missense,1,17,54,214,287,0.014167,0.054000,0.152857,0.179375


### Step 2: weighted score

In [30]:
weights = np.array([0.25, 0.5, 0.75, 1.0])
toy["weighted_score"] = freq.dot(weights) / freq.sum(axis = 1)  #dot product of rows with weights yield scalars for row. sum of weighted frequencies divided by sum of all frequencies
toy

,variant,variant_type,replicate,bin1_count,bin2_count,bin3_count,bin4_count,bin1_freq,bin2_freq,bin3_freq,bin4_freq,weighted_score
0,WT,wild_type,1,9,42,267,376,0.007500,0.042000,0.190714,0.235000,0.843642
1,p.Gly12Gly,synonymous,1,12,32,247,316,0.010000,0.032000,0.176429,0.197500,0.837455
2,p.Trp18Ter,nonsense,1,221,110,41,6,0.184167,0.110000,0.029286,0.003750,0.387393
3,p.Gln36Ter,nonsense,1,217,115,38,11,0.180833,0.115000,0.027143,0.006875,0.393937
4,p.Tyr52Ter,nonsense,1,207,89,30,7,0.172500,0.089000,0.021429,0.004375,0.376158
5,p.Leu25Pro,missense,1,190,154,68,11,0.158333,0.154000,0.048571,0.006875,0.434735
6,p.Val30Asp,missense,1,124,208,146,36,0.103333,0.208000,0.104286,0.022500,0.526221
7,p.Ile40Asn,missense,1,40,172,258,43,0.033333,0.172000,0.184286,0.026875,0.622872
8,p.Gly46Arg,missense,1,159,17,22,181,0.132500,0.017000,0.015714,0.113125,0.598319
9,p.Ser45Thr,missense,1,17,54,214,287,0.014167,0.054000,0.152857,0.179375,0.810591


### Check your work
Pick one row and compute its `W` by hand or with a calculator. Compare it with your code's answer.

In [ ]:
print("numerator", sum(toy.loc[1, freq_cols] * weights))
print("denominator", sum(toy.loc[1, freq_cols]))
print("first weighted score", sum(toy.loc[1, freq_cols] * weights) /  sum(toy.loc[1, freq_cols])) 

numerator 0.34832142857142856
denominator 0.4159285714285714
first weighted score 0.8374549201442555


**Presentation notes**
- What I did:
- How the code works:
- How I checked:

## Aim 3. Compute normalized scores for each variant

Reference:

$$\text{score} = \frac{W - W_{\text{nonsense, median}}}{W_{\text{WT}} - W_{\text{nonsense, median}}}$$

**Decision to record:** the two replicates have different bin totals. Do you compute the nonsense and WT references separately within each replicate, or once across both? Write down your choice and why.

*Your choice and reasoning:* seperately across replicates because they represent different sequencing experiments

### Step 1: nonsense reference

In [58]:
nonsense_W_median = toy[toy.variant_type == "nonsense"].groupby("replicate")["weighted_score"].median() #median of nonsense variant weighted scores
nonsense_W_median

replicate
1    0.387393
2    0.403076
Name: weighted_score, dtype: float64

### Step 2: wild-type reference

In [59]:
wt_W = toy[toy.variant_type == "wild_type"].groupby("replicate")["weighted_score"].first() #get the first row in each grouped df (wild type) for normalize
wt_W


replicate
1    0.843642
2    0.844702
Name: weighted_score, dtype: float64

### Step 3: normalized score

In [68]:
toy["abundance_score"] = (toy["weighted_score"] - toy["replicate"].map(nonsense_W_median)) / (toy["replicate"].map(wt_W) - toy["replicate"].map(nonsense_W_median))
toy

,variant,variant_type,replicate,bin1_count,bin2_count,bin3_count,bin4_count,bin1_freq,bin2_freq,bin3_freq,bin4_freq,weighted_score,abundance_score
0,WT,wild_type,1,9,42,267,376,0.007500,0.042000,0.190714,0.235000,0.843642,1.000000
1,p.Gly12Gly,synonymous,1,12,32,247,316,0.010000,0.032000,0.176429,0.197500,0.837455,0.986439
2,p.Trp18Ter,nonsense,1,221,110,41,6,0.184167,0.110000,0.029286,0.003750,0.387393,0.000000
3,p.Gln36Ter,nonsense,1,217,115,38,11,0.180833,0.115000,0.027143,0.006875,0.393937,0.014342
4,p.Tyr52Ter,nonsense,1,207,89,30,7,0.172500,0.089000,0.021429,0.004375,0.376158,-0.024626
5,p.Leu25Pro,missense,1,190,154,68,11,0.158333,0.154000,0.048571,0.006875,0.434735,0.103764
6,p.Val30Asp,missense,1,124,208,146,36,0.103333,0.208000,0.104286,0.022500,0.526221,0.304282
7,p.Ile40Asn,missense,1,40,172,258,43,0.033333,0.172000,0.184286,0.026875,0.622872,0.516120
8,p.Gly46Arg,missense,1,159,17,22,181,0.132500,0.017000,0.015714,0.113125,0.598319,0.462305
9,p.Ser45Thr,missense,1,17,54,214,287,0.014167,0.054000,0.152857,0.179375,0.810591,0.927559


### Check your work
What value should the wild-type row have after normalizing? What about a variant sitting exactly at the median nonsense level? Test both.

In [65]:
#i have no idea how to do this

### Combining replicates (Session 2 discussion will ask about this)
How did you treat the two replicates? Did you keep them separate, average them, or something else? Do they agree with each other?

In [ ]:
# we treated them seperately

**Presentation notes**
- What I did:
- How the code works:
- How I checked:

## Aim 4. Think critically about the results

Prompts (Session 2's discussion asks about the first two):
- Which variants look most damaging? Which look most like wild type?
- Did anything surprise you? How would you explain it?
- Does a single score hide anything? (Could two variants with different bin distributions get the same `W`?)
- What can invented data and a two-replicate design tell us, and what can they not?

In [ ]:
# Optional: sort or select rows to support what you write below

*Your observations:*

## Reproducibility record

Run this last, after restarting the kernel and running all cells. If `session_info` isn't installed, install it in the same environment as this notebook first.

In [ ]:
import session_info
session_info.show(dependencies=True)